In [1]:
import os
import pandas as pd
import joblib
import chromadb

from pathlib import Path
from dotenv import load_dotenv
from google import genai
from sentence_transformers import SentenceTransformer

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
from dotenv import load_dotenv
import os

env_file = r"C:\Packaging_AI_Control_Tower\.env"

load_dotenv(env_file, override=True)

api_key = os.getenv("GEMINI_API_KEY")

if api_key:
    print("Gemini API key loaded successfully")
else:
    print("Gemini API key NOT found")

Gemini API key loaded successfully


In [3]:
from google import genai

client = genai.Client(
    api_key=api_key
)

print("Gemini client created successfully")

Gemini client created successfully


In [4]:
response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents="Explain machine downtime in a packaging industry in simple words."
)

print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In the packaging industry, **machine downtime** is simply any period when your packaging line has stopped working and is not producing finished products.

Think of it like a **traffic jam** on a conveyor belt. If the machines aren't running, money is being lost because you aren't packing anything to sell.

Here is a breakdown of why it happens and why it matters:

---

### 1. The Two Types of Downtime
*   **Planned Downtime:** This is when you *choose* to turn the machines off. Examples include:
    *   **Changeovers:** Switching the machine from packing 10oz bags of chips to 20oz bags.
    *   **Scheduled Maintenance:** Cleaning or oiling the machines to prevent them from breaking later.
*   **Unplanned Downtime:** This is the "bad" kind. It happens when you don't expect it. Examples include:
    *   **Mechanical Failures:** A motor burns out or a belt snaps.
    *   **Material Issues:** The packaging film keeps tearing, or the cardboard boxes are jammed.
    *   **Human Error:** An o

In [5]:
data_path = r"C:\Packaging_AI_Control_Tower\Data\processed\features.csv"

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (23376, 170)
        interval_start equipment_ID  count_sum  A_028  A_029  A_024  A_045  \
0  2020-01-01 14:00:00          s_1          4    0.0      0      0    NaN   
1  2020-01-01 15:00:00          s_1          2    0.0      0      0    NaN   
2  2020-01-01 17:00:00          s_1          1    0.0      0      0    NaN   
3  2020-01-01 18:00:00          s_1          4    0.0      0      0    NaN   
4  2020-01-01 19:00:00          s_1          1    0.0      0      0    NaN   

   A_001  A_058  A_064  ...  A_053  A_054  A_060  A_061  \
0    0.0    NaN    0.0  ...    NaN    NaN    NaN    NaN   
1    0.0    NaN    0.0  ...    NaN    NaN    NaN    NaN   
2    0.0    NaN    0.0  ...    NaN    NaN    NaN    NaN   
3    0.0    NaN    0.0  ...    NaN    NaN    NaN    NaN   
4    0.0    NaN    0.0  ...    NaN    NaN    NaN    NaN   

   production_rolling_6h  downtime_rolling_6h  production_change  \
0               0.861729             0.052261           0.000000   
1           

In [6]:
anomaly_path = r"C:\Packaging_AI_Control_Tower\Data\processed\anomaly_results.csv"

anomaly_df = pd.read_csv(anomaly_path)

print("Anomaly dataset shape:", anomaly_df.shape)
print(anomaly_df.head())

Anomaly dataset shape: (23376, 174)
        interval_start equipment_ID  count_sum  A_028  A_029  A_024  A_045  \
0  2020-01-01 14:00:00          s_1          4    0.0      0      0    NaN   
1  2020-01-01 15:00:00          s_1          2    0.0      0      0    NaN   
2  2020-01-01 17:00:00          s_1          1    0.0      0      0    NaN   
3  2020-01-01 18:00:00          s_1          4    0.0      0      0    NaN   
4  2020-01-01 19:00:00          s_1          1    0.0      0      0    NaN   

   A_001  A_058  A_064  ...  production_rolling_6h  downtime_rolling_6h  \
0    0.0    NaN    0.0  ...               0.861729             0.052261   
1    0.0    NaN    0.0  ...               0.866313             0.084982   
2    0.0    NaN    0.0  ...               0.903703             0.063827   
3    0.0    NaN    0.0  ...               0.795544             0.164966   
4    0.0    NaN    0.0  ...               0.833840             0.133529   

   production_change  downtime_change  perfo

In [7]:
risk_path = r"C:\Packaging_AI_Control_Tower\Data\processed\risk_results.csv"

risk_df = pd.read_csv(risk_path)

print("Risk dataset shape:", risk_df.shape)
print(risk_df.head())

Risk dataset shape: (4670, 177)
        interval_start equipment_ID  count_sum  A_028  A_029  A_024  A_045  \
0  2021-09-29 10:00:00          s_2          3    0.0      0      0    0.0   
1  2021-09-29 10:00:00          s_5          1    NaN      0      0    0.0   
2  2021-09-29 11:00:00          s_5          3    NaN      0      0    0.0   
3  2021-09-29 11:00:00          s_3          3    0.0      0      0    0.0   
4  2021-09-29 12:00:00          s_3          3    0.0      0      0    0.0   

   A_001  A_058  A_064  ...  downtime_change  performance_change  \
0    1.0    0.0    NaN  ...        -0.366543            0.009372   
1    NaN    NaN    NaN  ...         0.017514            0.001534   
2    NaN    NaN    NaN  ...         0.028751            0.169927   
3    2.0    0.0    0.0  ...         0.024945           -0.025711   
4    1.0    0.0    0.0  ...         0.166011           -0.033187   

   health_score  future_production  future_downtime  future_performance_loss  \
0     99.1

In [8]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully


In [9]:
chroma_client = chromadb.PersistentClient(
    path=r"C:\Packaging_AI_Control_Tower\data\chroma_db"
)

collection = chroma_client.get_or_create_collection(
    name="packaging_knowledge"
)

print("RAG knowledge base connected")

RAG knowledge base connected


In [10]:
def search_knowledge(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query]
    )[0]

    results = collection.query(
        query_embeddings=[
            query_embedding.tolist()
        ],
        n_results=top_k
    )

    return results

In [11]:
results = search_knowledge(
    "machine has high downtime and repeated alarms"
)

print(results["documents"][0])

['c component has failed.\n\n2. ALARM INVESTIGATION\n\nWhen an alarm is detected, the control tower should examine:\n\n- Equipment ID\n- Alarm identifier\n- Alarm occurrence time\n- Alarm duration when available\n- Machine production around the event\n- Downtime around the event\n- Performance loss around the event\n- Previous alarms\n- Anomaly detection results\n- Risk prediction results\n\n3. RECENT ALARM ANALYSIS\n\nThe system should determine whether the alarm is:\n\n- A single isolated event\n- Repeated over a short period\n- Associated with production reduction\n- Associated with increased downtime\n- Associated with an anomaly\n- Associated with elevated predicted risk\n\nRepeated events may require additional investigation.\n\n4. ALARM FREQUENCY\n\nA high alarm count may indicate repeated operational events.\n\nHowev', 'estigation.\n\n4. ALARM FREQUENCY\n\nA high alarm count may indicate repeated operational events.\n\nHowever, alarm frequency should not automatically be interp

In [12]:
def get_machine_data(equipment_id):

    result = df[
        df["equipment_ID"].astype(str) == str(equipment_id)
    ].copy()

    return result

In [13]:
machine_data = get_machine_data("M001")

print(machine_data.head())

Empty DataFrame
Columns: [interval_start, equipment_ID, count_sum, A_028, A_029, A_024, A_045, A_001, A_058, A_064, A_030, A_101, A_002, A_107, A_056, A_015, A_032, A_026, A_003, A_008, A_078, A_070, A_004, A_005, A_009, A_016, A_010, A_027, A_020, A_017, A_048, A_022, A_011, A_012, A_013, A_042, A_023, A_034, A_036, A_079, A_077, A_090, A_021, A_049, A_006, #changes, %idle, %production, %downtime, %performance_loss, %scheduled_downtime, idle/idle, idle/production, idle/downtime, idle/performance_loss, idle/scheduled_downtime, production/idle, production/production, production/downtime, production/performance_loss, production/scheduled_downtime, downtime/idle, downtime/production, downtime/downtime, downtime/performance_loss, downtime/scheduled_downtime, performance_loss/idle, performance_loss/production, performance_loss/downtime, performance_loss/performance_loss, performance_loss/scheduled_downtime, scheduled_downtime/idle, scheduled_downtime/production, scheduled_downtime/downtime,

In [14]:
print(df["equipment_ID"].unique()[:20])

<ArrowStringArray>
['s_1', 's_2', 's_3', 's_4', 's_5']
Length: 5, dtype: str


In [15]:
def get_machine_anomaly(equipment_id):

    result = anomaly_df[
        anomaly_df["equipment_ID"].astype(str) == str(equipment_id)
    ].copy()

    return result

In [16]:
anomaly_data = get_machine_anomaly("M001")

print(anomaly_data.head())

Empty DataFrame
Columns: [interval_start, equipment_ID, count_sum, A_028, A_029, A_024, A_045, A_001, A_058, A_064, A_030, A_101, A_002, A_107, A_056, A_015, A_032, A_026, A_003, A_008, A_078, A_070, A_004, A_005, A_009, A_016, A_010, A_027, A_020, A_017, A_048, A_022, A_011, A_012, A_013, A_042, A_023, A_034, A_036, A_079, A_077, A_090, A_021, A_049, A_006, #changes, %idle, %production, %downtime, %performance_loss, %scheduled_downtime, idle/idle, idle/production, idle/downtime, idle/performance_loss, idle/scheduled_downtime, production/idle, production/production, production/downtime, production/performance_loss, production/scheduled_downtime, downtime/idle, downtime/production, downtime/downtime, downtime/performance_loss, downtime/scheduled_downtime, performance_loss/idle, performance_loss/production, performance_loss/downtime, performance_loss/performance_loss, performance_loss/scheduled_downtime, scheduled_downtime/idle, scheduled_downtime/production, scheduled_downtime/downtime,

In [17]:
def get_machine_risk(equipment_id):

    result = risk_df[
        risk_df["equipment_ID"].astype(str) == str(equipment_id)
    ].copy()

    return result

In [18]:
risk_data = get_machine_risk("M001")

print(risk_data.head())

Empty DataFrame
Columns: [interval_start, equipment_ID, count_sum, A_028, A_029, A_024, A_045, A_001, A_058, A_064, A_030, A_101, A_002, A_107, A_056, A_015, A_032, A_026, A_003, A_008, A_078, A_070, A_004, A_005, A_009, A_016, A_010, A_027, A_020, A_017, A_048, A_022, A_011, A_012, A_013, A_042, A_023, A_034, A_036, A_079, A_077, A_090, A_021, A_049, A_006, #changes, %idle, %production, %downtime, %performance_loss, %scheduled_downtime, idle/idle, idle/production, idle/downtime, idle/performance_loss, idle/scheduled_downtime, production/idle, production/production, production/downtime, production/performance_loss, production/scheduled_downtime, downtime/idle, downtime/production, downtime/downtime, downtime/performance_loss, downtime/scheduled_downtime, performance_loss/idle, performance_loss/production, performance_loss/downtime, performance_loss/performance_loss, performance_loss/scheduled_downtime, scheduled_downtime/idle, scheduled_downtime/production, scheduled_downtime/downtime,

In [19]:
def get_knowledge(query, top_k=5):

    results = search_knowledge(
        query,
        top_k=top_k
    )

    context_parts = []

    for i, document in enumerate(
        results["documents"][0]
    ):

        source = results["metadatas"][0][i]["source"]

        context_parts.append(
            f"""
SOURCE: {source}

CONTENT:
{document}
"""
        )

    return "\n\n".join(context_parts)

In [20]:
knowledge = get_knowledge(
    "high downtime repeated alarms investigation"
)

print(knowledge)


SOURCE: ..\documents\alarm_guides\packaging_alarm_investigation.txt

CONTENT:
c component has failed.

2. ALARM INVESTIGATION

When an alarm is detected, the control tower should examine:

- Equipment ID
- Alarm identifier
- Alarm occurrence time
- Alarm duration when available
- Machine production around the event
- Downtime around the event
- Performance loss around the event
- Previous alarms
- Anomaly detection results
- Risk prediction results

3. RECENT ALARM ANALYSIS

The system should determine whether the alarm is:

- A single isolated event
- Repeated over a short period
- Associated with production reduction
- Associated with increased downtime
- Associated with an anomaly
- Associated with elevated predicted risk

Repeated events may require additional investigation.

4. ALARM FREQUENCY

A high alarm count may indicate repeated operational events.

Howev



SOURCE: ..\documents\alarm_guides\packaging_alarm_investigation.txt

CONTENT:
estigation.

4. ALARM FREQUENCY

A high

In [21]:
def investigate_machine(equipment_id, question):

    # 1. Get operational data
    machine_data = get_machine_data(equipment_id)

    # 2. Get anomaly information
    anomaly_data = get_machine_anomaly(equipment_id)

    # 3. Get risk information
    risk_data = get_machine_risk(equipment_id)

    # 4. Retrieve knowledge
    knowledge = get_knowledge(question)

    return {
        "machine_data": machine_data,
        "anomaly_data": anomaly_data,
        "risk_data": risk_data,
        "knowledge": knowledge
    }

In [22]:
investigation = investigate_machine(
    "M001",
    "Machine has high downtime and repeated alarms. What should be investigated?"
)

print("Machine Data:")
print(investigation["machine_data"].tail())

print("\nAnomaly Data:")
print(investigation["anomaly_data"].tail())

print("\nRisk Data:")
print(investigation["risk_data"].tail())

print("\nKnowledge:")
print(investigation["knowledge"])

Machine Data:
Empty DataFrame
Columns: [interval_start, equipment_ID, count_sum, A_028, A_029, A_024, A_045, A_001, A_058, A_064, A_030, A_101, A_002, A_107, A_056, A_015, A_032, A_026, A_003, A_008, A_078, A_070, A_004, A_005, A_009, A_016, A_010, A_027, A_020, A_017, A_048, A_022, A_011, A_012, A_013, A_042, A_023, A_034, A_036, A_079, A_077, A_090, A_021, A_049, A_006, #changes, %idle, %production, %downtime, %performance_loss, %scheduled_downtime, idle/idle, idle/production, idle/downtime, idle/performance_loss, idle/scheduled_downtime, production/idle, production/production, production/downtime, production/performance_loss, production/scheduled_downtime, downtime/idle, downtime/production, downtime/downtime, downtime/performance_loss, downtime/scheduled_downtime, performance_loss/idle, performance_loss/production, performance_loss/downtime, performance_loss/performance_loss, performance_loss/scheduled_downtime, scheduled_downtime/idle, scheduled_downtime/production, scheduled_down

In [26]:
machine_data_text = investigation["machine_data"].tail(10).to_string(
    index=False
)

anomaly_text = investigation["anomaly_data"].tail(10).to_string(
    index=False
)

risk_text = investigation["risk_data"].tail(10).to_string(
    index=False
)

knowledge_text = investigation["knowledge"]

In [28]:
equipment_id = "M001"

agent_prompt = f"""
You are an AI Operations Control Tower Agent
for a packaging industry.

USER QUESTION:
Why is machine {equipment_id} experiencing
high downtime and repeated alarms?

OPERATIONAL DATA:
{machine_data_text}

ANOMALY DETECTION RESULTS:
{anomaly_text}

RISK PREDICTION RESULTS:
{risk_text}

RETRIEVED KNOWLEDGE:
{knowledge_text}

Instructions:

1. Analyze the available evidence.

2. Clearly separate observed facts from possible causes.

3. Do not invent the meaning of alarm codes.

4. Do not claim a possible cause is confirmed.

5. Use the retrieved knowledge as project-specific
   guidance.

6. Recommend practical investigation steps.

7. Physical machine intervention requires
   human verification.

Return:

Incident Summary:
Observed Evidence:
Anomaly Status:
Risk Status:
Possible Root Cause:
Confidence:
Recommended Investigation:
Human Verification:
"""

In [29]:
response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=agent_prompt
)

print(response.text)

**Incident Summary:**
The request concerns reports of high downtime and repeated alarms for machine M001. Current system logs and data feeds for this equipment are returning empty datasets, preventing a real-time correlation between the alarms and operational performance metrics.

**Observed Evidence:**
*   **Reported Issue:** High downtime and repeated alarms on machine M001.
*   **Data Availability:** Operational, anomaly, and risk prediction data streams for M001 are currently returning null/empty values.
*   **Evidence Correlation:** Due to the missing data, it is currently impossible to determine if the alarms coincide with production decreases or performance loss.

**Anomaly Status:**
*   **Unknown:** Anomaly detection results are currently unavailable/empty.

**Risk Status:**
*   **Unknown:** Risk prediction results are currently unavailable/empty.

**Possible Root Cause:**
*   **Indeterminate:** Without access to the specific alarm codes and correlating downtime logs, a root-ca

In [32]:
def run_agent(equipment_id, question):

    # Get machine data
    machine_data = get_machine_data(equipment_id)

    # Get anomaly data
    anomaly_data = get_machine_anomaly(equipment_id)

    # Get risk data
    risk_data = get_machine_risk(equipment_id)

    # Get RAG knowledge
    knowledge = get_knowledge(question)

    # Convert data to text
    machine_data_text = machine_data.tail(10).to_string(
        index=False
    )

    anomaly_text = anomaly_data.tail(10).to_string(
        index=False
    )

    risk_text = risk_data.tail(10).to_string(
        index=False
    )

    # Create Agent prompt
    agent_prompt = f"""
You are an AI Operations Control Tower Agent
for a packaging industry.

USER QUESTION:
{question}

MACHINE:
{equipment_id}

OPERATIONAL DATA:
{machine_data_text}

ANOMALY DETECTION RESULTS:
{anomaly_text}

RISK PREDICTION RESULTS:
{risk_text}

RETRIEVED KNOWLEDGE:
{knowledge}

INSTRUCTIONS:

1. Analyze the available evidence.
2. Separate observed facts from possible causes.
3. Do not invent alarm meanings.
4. Do not claim a possible cause is confirmed.
5. Use retrieved knowledge as project-specific guidance.
6. Recommend practical investigation steps.
7. Physical machine intervention requires human verification.

Return:

Incident Summary:
Observed Evidence:
Anomaly Status:
Risk Status:
Possible Root Cause:
Confidence:
Recommended Investigation:
Human Verification:
"""

    # Ask Gemini
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=agent_prompt
    )

    return response.text

In [33]:
answer = run_agent(
    "M001",
    "Machine has high downtime and repeated alarms. What should be investigated?"
)

print(answer)

**Incident Summary:**
Machine M001 is currently experiencing reports of high downtime and repeated alarms. Analysis of the available operational, anomaly, and risk data indicates that the system is currently unable to provide specific telemetry or automated pattern identification for this machine.

**Observed Evidence:**
*   **Reported Status:** High downtime and repeated alarm occurrences on machine M001.
*   **Data Availability:** The provided data frames (Operational Data, Anomaly Detection, and Risk Prediction) are empty, preventing automated correlation of specific alarm codes with downtime events.

**Anomaly Status:**
No anomaly data is available in the current system report to confirm or characterize the deviation.

**Risk Status:**
No risk prediction data is available in the current system report.

**Possible Root Cause:**
The root cause cannot be determined with the current data set. Potential areas contributing to high downtime (per maintenance guidelines) include:
*   Operat